# End-to-end pipeline: data → preprocessing → model → evaluation

In [1]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "preprocessing" / "align.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. Data

In [2]:
from src.data import TedData

ted = TedData()

## 2. Preprocessing

In [3]:
SRC_LANG, TGT_LANG = "en", "az"
DIRECTION = f"{SRC_LANG}-{TGT_LANG}"

train_pairs = [(s, t) for _, s, t in ted.pairs("train", SRC_LANG, TGT_LANG)][:1000]
dev_pairs = [(s, t) for _, s, t in ted.pairs("dev", SRC_LANG, TGT_LANG)][:100]
test_pairs = [(s, t) for _, s, t in ted.pairs("test", SRC_LANG, TGT_LANG)][:100]
len(train_pairs), len(dev_pairs), len(test_pairs)

(1000, 100, 100)

## 3. Model

In [4]:
from src.model import Vocabulary, build_model, translate_corpus, save_translations

MODEL_NAME = "seq2seq"
SYSTEM = MODEL_NAME
MAX_LENGTH = {"seq2seq": 20, "copy": 20, "nllb": 128}[MODEL_NAME]

all_pairs = train_pairs + dev_pairs + test_pairs
src_vocab = Vocabulary([s for s, _ in all_pairs])
tgt_vocab = Vocabulary([t for _, t in all_pairs])

model = build_model(MODEL_NAME, src_vocab, tgt_vocab, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model.fit(pairs=list(train_pairs), epochs=10)

  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:20<03:04, 20.53s/it]

 20%|██        | 2/10 [00:41<02:46, 20.76s/it]

 30%|███       | 3/10 [01:01<02:23, 20.47s/it]

 40%|████      | 4/10 [01:21<02:00, 20.15s/it]

...

 70%|███████   | 7/10 [02:38<01:15, 25.12s/it]

 80%|████████  | 8/10 [03:12<00:56, 28.08s/it]

 90%|█████████ | 9/10 [03:45<00:29, 29.75s/it]

100%|██████████| 10/10 [04:13<00:00, 29.13s/it]

100%|██████████| 10/10 [04:13<00:00, 25.37s/it]

## 4. Evaluation

In [5]:
from src.evaluation import evaluate_direction

hyps = translate_corpus(model, [s for s, _ in test_pairs], max_length=MAX_LENGTH)
out_dir = ROOT / "results" / "translations" / SYSTEM
save_translations(out_dir, DIRECTION, [s for s, _ in test_pairs], [t for _, t in test_pairs], hyps)

evaluate_direction(out_dir, DIRECTION, "bleu")

  0%|          | 0/100 [00:00<?, ?it/s]

 23%|██▎       | 23/100 [00:00<00:00, 224.87it/s]

 46%|████▌     | 46/100 [00:00<00:00, 186.73it/s]

 66%|██████▌   | 66/100 [00:00<00:00, 166.03it/s]

 83%|████████▎ | 83/100 [00:00<00:00, 166.86it/s]

100%|██████████| 100/100 [00:00<00:00, 174.66it/s]

{'direction': 'en-az',
 'metric': 'bleu',
 'n_sentences': 100,
 'score': 0.6796353311603136,
 'signature': 'nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0'}